In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
model = load_model('model.h5')

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [35]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [36]:
input_df = pd.DataFrame([input_data])

In [37]:
input_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [38]:
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()

In [39]:
geo_encoded_df = pd.DataFrame(geo_encoded , columns = onehot_encoder_geo.get_feature_names_out(['Geography']))

In [40]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [41]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [42]:
input_df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [43]:
input_df = pd.concat([input_df.drop('Geography',axis=1) , geo_encoded_df], axis = 1)

In [45]:
scaled_input = scaler.transform(input_df)

In [49]:
prediction = model.predict(scaled_input)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


array([[0.02568714]], dtype=float32)

In [50]:
prediction_prob = prediction[0][0]

In [51]:
prediction_prob

np.float32(0.02568714)

In [52]:
if prediction_prob > 0.5 :
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")

The customer is not likely to churn
